# Mobile-FOG (Bagian Rhama): Ekstraksi `YOLO26n-Pose` + `PatientLock-2Pass` & Preprocessing Scenario A / B / D
**Target Lingkungan:** Kaggle Notebook — **Accelerator: GPU T4 x2** (`2× NVIDIA T4`) & **Internet: ON**

Notebook ini mengintegrasikan **bounding box khusus pasien (`ajikajadeh/mobile-fog-patient-bboxes`)** untuk menghilangkan masalah *ID switching* ke pengawas (`97,19%` frame multi-person → *off-patient lock rate* turun dari `~30%` ke `0,40%`, 100% *35-Fold LOSO-safe*):
1. **Isolasi Pasien + Ekstraksi `YOLO26n-Pose` Paralel (`2× T4`)**: *Crop* frame mengikuti `patient_box_norm` + *padding* `15%` + *masking* area pengawas (tanpa menghapus area inti pasien) → `YOLO26n-pose` → **IoU matching + Remap** ke koordinat asli `1280×720`.
2. **Preprocessing Scenario A (`A_baseline_norm`)**: Topologi `LB6` (`L/R Hip, Knee, Ankle`), normalisasi pelvis & panjang kaki, interpolasi `conf < 0.2`, `7` kanal.
3. **Preprocessing Scenario B (`B_denoised_hampel_accel`)**: *Hampel rolling-median* + *Butterworth 9 Hz* (di `30 Hz`) + akselerasi → `9` kanal.
4. **Preprocessing Scenario D (`D_windowed_stgcn_cls`)**: Jendela `2 detik` (`15 Hz` & `30 Hz`), filter `inside_imu_mask == 1`, dan `StandardScaler` hanya dari `train`.


In [ ]:
# Install / upgrade ultralytics agar mendukung YOLO26-pose
!pip install -q --upgrade ultralytics scipy

import os
import json
import time
import torch
import ultralytics
import numpy as np
import pandas as pd
from pathlib import Path

print(f"PyTorch     : {torch.__version__}")
print(f"Ultralytics : {ultralytics.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)")


In [ ]:
# ==============================================================================
# TEMPAT MELETAKKAN PATH DATASET KAGGLE (EDIT BAGIAN INI)
# ==============================================================================
# 1) Folder 'Train Val Test' yang berisi subfolder: train/, val/, test/
DATASET_ROOT = Path("/kaggle/input/GANTI_DENGAN_SLUG_DATASET_VIDEO/Train Val Test")

# 2) Folder dataset 'ajikajadeh/mobile-fog-patient-bboxes' (berisi patient_bboxes_npz/)
BBOX_ROOT = Path("/kaggle/input/mobile-fog-patient-bboxes")

# 3) Folder output baru di Kaggle Working Directory
OUTPUT_ROOT = Path("/kaggle/working/mobilefog_rhama_patientlock")

# Auto-detect fallback untuk DATASET_ROOT
if not (DATASET_ROOT / "train" / "videos").exists():
    detected = None
    for root in [Path("/kaggle/input"), Path("Data"), Path(".")]:
        if not root.exists():
            continue
        for cand in [root, *root.rglob("*")]:
            if cand.is_dir() and (cand / "train" / "videos").exists() and (cand / "val" / "videos").exists():
                detected = cand
                break
        if detected is not None:
            break
    if detected is not None:
        print(f"[AUTO-DETECT] DATASET_ROOT ditemukan otomatis di: {detected}")
        DATASET_ROOT = detected
    else:
        raise FileNotFoundError(f"Folder dataset video tidak ditemukan di {DATASET_ROOT}.")

# Auto-detect fallback untuk BBOX_ROOT
if not (BBOX_ROOT / "patient_bboxes_npz" / "train").exists():
    detected_bb = None
    for root in [Path("/kaggle/input"), Path("Data"), Path(".")]:
        if not root.exists():
            continue
        for cand in [root, *root.rglob("patient_bboxes_npz")]:
            if cand.is_dir() and (cand / "train").exists() and (cand / "test").exists():
                detected_bb = cand.parent
                break
        if detected_bb is not None:
            break
    if detected_bb is not None:
        print(f"[AUTO-DETECT] BBOX_ROOT ditemukan otomatis di: {detected_bb}")
        BBOX_ROOT = detected_bb
    else:
        raise FileNotFoundError(
            f"Folder 'patient_bboxes_npz' tidak ditemukan di {BBOX_ROOT}. "
            "Tambahkan dataset Kaggle 'ajikajadeh/mobile-fog-patient-bboxes' ke Input notebook."
        )

# ==============================================================================
# KONFIGURASI MODEL, PATIENTLOCK CROP, & PREPROCESSING (SESUAI TUGAS RHAMA)
# ==============================================================================
YOLO_WEIGHTS = "yolo26n-pose.pt"
IMG_SIZE = 640
DET_CONF = 0.20                    # Sedikit lebih longgar di dalam crop pasien agar kaki tidak hilang
BATCH_SIZE = 32

# Parameter PatientLock-2Pass Crop & Masking
CROP_PAD_RATIO = 0.15              # Padding 15% di sekitar bbox pasien agar engkel tidak terpotong
MASK_SUPERVISORS = True            # Tutup area pengawas (non_patient_flat) di tepi crop dengan abu netral
MIN_CROP_IOU = 0.20                # IoU minimum antara deteksi YOLO dalam crop vs bbox inti pasien

# Topologi LB6: 6 sendi tungkai bawah dari 17 COCO keypoints
LB6_INDICES = [11, 12, 13, 14, 15, 16]
LB6_NAMES = ["L_Hip", "R_Hip", "L_Knee", "R_Knee", "L_Ankle", "R_Ankle"]

CONF_VALID_THRES = 0.20
HAMPEL_WINDOW = 7
HAMPEL_SIGMA = 3.0
BUTTER_CUTOFF_HZ = 9.0

WINDOW_SEC = 2.0
STRIDE_SEC = 1.0
RANDOM_SEED = 42

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"DATASET_ROOT : {DATASET_ROOT}")
print(f"BBOX_ROOT    : {BBOX_ROOT}")
print(f"OUTPUT_ROOT  : {OUTPUT_ROOT}")


In [ ]:
SPLITS = ["train", "val", "test"]
trial_records = []

for split in SPLITS:
    v_dir = DATASET_ROOT / split / "videos"
    l30_dir = DATASET_ROOT / split / "labels_30hz"
    l15_dir = DATASET_ROOT / split / "labels_15hz"
    bb_dir = BBOX_ROOT / "patient_bboxes_npz" / split

    videos = sorted(v_dir.glob("*.mp4"))
    for vp in videos:
        tid = vp.stem
        sid = tid.split("_")[0].replace("PDFE", "")
        l30 = l30_dir / f"{tid}.csv"
        l15 = l15_dir / f"{tid}.csv"
        bb_npz = bb_dir / f"{tid}.npz"
        assert l30.exists(), f"Missing 30Hz label: {l30}"
        assert l15.exists(), f"Missing 15Hz label: {l15}"
        assert bb_npz.exists(), f"Missing PatientLock bbox: {bb_npz}"
        trial_records.append({
            "split": split,
            "trial_id": tid,
            "subject_id": sid,
            "video_path": str(vp),
            "label30_path": str(l30),
            "label15_path": str(l15),
            "bbox_npz_path": str(bb_npz),
        })

df_manifest = pd.DataFrame(trial_records)
df_manifest.to_csv(OUTPUT_ROOT / "manifest_trials_local.csv", index=False)

print("Ringkasan Dataset (Video + Label 15/30Hz + PatientLock BBox):")
print(df_manifest.groupby("split").agg(trials=("trial_id", "count"), subjects=("subject_id", "nunique")))
assert len(df_manifest) == 71, f"Peringatan: ditemukan {len(df_manifest)} video (harapan 71)."
print("Audit 71 video, label 15Hz/30Hz, & patient_bboxes_npz: LENGKAP.")


In [ ]:
# ==============================================================================
# TAHAP 1: EKSTRAKSI YOLO26n-POSE + PATIENTLOCK-2PASS PARALEL PADA 2x T4 GPU
# ==============================================================================
import sys
import subprocess
from ultralytics import YOLO

RAW_POSE_DIR = OUTPUT_ROOT / "01_extracted_poses_raw" / "yolo26n_rhama"
RAW_POSE_DIR.mkdir(parents=True, exist_ok=True)

# Pre-download weights sekali di main process agar 2 worker tidak balapan download
_ = YOLO(YOLO_WEIGHTS)

joint_schema = {
    "estimator": YOLO_WEIGHTS,
    "patient_lock": "PatientLock-2Pass (ajikajadeh/mobile-fog-patient-bboxes)",
    "crop_pad_ratio": CROP_PAD_RATIO,
    "mask_supervisors": MASK_SUPERVISORS,
    "num_keypoints": 17,
    "coco_names": [
        "nose", "left_eye", "right_eye", "left_ear", "right_ear",
        "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
        "left_wrist", "right_wrist", "left_hip", "right_hip",
        "left_knee", "right_knee", "left_ankle", "right_ankle",
    ],
    "lb6_indices": LB6_INDICES,
    "lb6_names": LB6_NAMES,
}
with open(RAW_POSE_DIR.parent / "joint_schema.json", "w") as f:
    json.dump(joint_schema, f, indent=2)

# Tulis skrip worker terisolasi (bebas error pickle multiprocessing di Jupyter)
worker_script = OUTPUT_ROOT / "_yolo_patientlock_worker.py"
worker_script.write_text("""import json
import sys
from pathlib import Path
import cv2
import numpy as np
import torch
from ultralytics import YOLO


def _box_iou_one_to_many(target_box: np.ndarray, boxes: np.ndarray) -> np.ndarray:
    ix1 = np.maximum(target_box[0], boxes[:, 0])
    iy1 = np.maximum(target_box[1], boxes[:, 1])
    ix2 = np.minimum(target_box[2], boxes[:, 2])
    iy2 = np.minimum(target_box[3], boxes[:, 3])
    inter = np.maximum(0.0, ix2 - ix1) * np.maximum(0.0, iy2 - iy1)
    area_t = max(0.0, (target_box[2] - target_box[0]) * (target_box[3] - target_box[1]))
    area_b = np.maximum(0.0, (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]))
    return inter / np.maximum(area_t + area_b - inter, 1e-6)


def _crop_and_mask_patient(
    frame: np.ndarray,
    pbox_norm: np.ndarray,
    sup_boxes_norm: np.ndarray,
    pad_ratio: float,
    mask_supervisors: bool,
):
    H, W = frame.shape[:2]
    px1 = float(np.clip(pbox_norm[0] * W, 0, W - 1))
    py1 = float(np.clip(pbox_norm[1] * H, 0, H - 1))
    px2 = float(np.clip(pbox_norm[2] * W, px1 + 1, W))
    py2 = float(np.clip(pbox_norm[3] * H, py1 + 1, H))

    bw = px2 - px1
    bh = py2 - py1
    cx1 = int(max(0, np.floor(px1 - pad_ratio * bw)))
    cy1 = int(max(0, np.floor(py1 - pad_ratio * bh)))
    cx2 = int(min(W, np.ceil(px2 + pad_ratio * bw)))
    cy2 = int(min(H, np.ceil(py2 + pad_ratio * bh)))

    crop = frame[cy1:cy2, cx1:cx2].copy()
    ch, cw = crop.shape[:2]
    core_in_crop = np.array([px1 - cx1, py1 - cy1, px2 - cx1, py2 - cy1], dtype=np.float32)

    if mask_supervisors and len(sup_boxes_norm) > 0:
        # Simpan area inti pasien dulu agar tidak ikut tertutup jika bbox pengawas sedikit bersinggungan
        ix1, iy1 = int(max(0, np.floor(core_in_crop[0]))), int(max(0, np.floor(core_in_crop[1])))
        ix2, iy2 = int(min(cw, np.ceil(core_in_crop[2]))), int(min(ch, np.ceil(core_in_crop[3])))
        patient_patch = crop[iy1:iy2, ix1:ix2].copy()

        for sb in sup_boxes_norm:
            sx1 = int(np.clip(np.floor(float(sb[0]) * W) - cx1, 0, cw))
            sy1 = int(np.clip(np.floor(float(sb[1]) * H) - cy1, 0, ch))
            sx2 = int(np.clip(np.ceil(float(sb[2]) * W) - cx1, 0, cw))
            sy2 = int(np.clip(np.ceil(float(sb[3]) * H) - cy1, 0, ch))
            if sx2 > sx1 and sy2 > sy1:
                crop[sy1:sy2, sx1:sx2] = (114, 114, 114)

        crop[iy1:iy2, ix1:ix2] = patient_patch

    pbox_px = np.array([px1, py1, px2, py2], dtype=np.float32)
    return crop, (cx1, cy1), core_in_crop, pbox_px


def _pick_pose_in_crop(result, offset_xy: tuple[int, int], core_in_crop: np.ndarray, pbox_px: np.ndarray, min_iou: float):
    if result.keypoints is None or result.boxes is None or len(result.boxes) == 0:
        return np.zeros((17, 3), dtype=np.float32), pbox_px, False

    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    kps_xy = result.keypoints.xy.cpu().numpy()
    if result.keypoints.conf is not None:
        kps_c = result.keypoints.conf.cpu().numpy()
    else:
        kps_c = np.ones((len(boxes_xyxy), 17), dtype=np.float32)

    ious = _box_iou_one_to_many(core_in_crop, boxes_xyxy)
    idx = int(np.argmax(ious))
    if ious[idx] < min_iou:
        return np.zeros((17, 3), dtype=np.float32), pbox_px, False

    # Remap koordinat dari crop kembali ke frame penuh (W, H)
    kp = np.concatenate([kps_xy[idx], kps_c[idx][:, None]], axis=-1).astype(np.float32)
    kp[:, 0] += float(offset_xy[0])
    kp[:, 1] += float(offset_xy[1])
    return kp, pbox_px, True


def main():
    job_path = Path(sys.argv[1])
    job = json.loads(job_path.read_text(encoding="utf-8"))
    gpu_label = job["gpu_label"]
    cfg = job["cfg"]
    tasks = job["tasks"]

    device_str = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = YOLO(cfg["weights"])
    out_dir = Path(cfg["out_dir"])

    for item in tasks:
        split = item["split"]
        tid = item["trial_id"]
        save_path = out_dir / split / f"{tid}.npz"
        save_path.parent.mkdir(parents=True, exist_ok=True)
        if save_path.exists():
            continue

        bb = np.load(item["bbox_npz_path"])
        pboxes_norm = bb["patient_box_norm"]
        is_det = bb["is_detected"]
        non_pat_flat = bb["non_patient_flat"]
        non_pat_off = bb["non_patient_offsets"]

        cap = cv2.VideoCapture(item["video_path"])
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 29.98)
        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n_bb = len(pboxes_norm)

        all_kps, all_boxes, all_valid = [], [], []
        batch_crops = []
        batch_meta = []  # (out_idx, offset_xy, core_in_crop, pbox_px)

        def flush_batch():
            if not batch_crops:
                return
            results = model.predict(
                batch_crops, imgsz=cfg["imgsz"], conf=cfg["det_conf"],
                device=device_str, verbose=False
            )
            for res, (o_idx, off_xy, core_box, pbox_px) in zip(results, batch_meta):
                kp, bx, val = _pick_pose_in_crop(res, off_xy, core_box, pbox_px, cfg["min_crop_iou"])
                all_kps[o_idx] = kp
                all_boxes[o_idx] = bx
                all_valid[o_idx] = val
            batch_crops.clear()
            batch_meta.clear()

        f_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            all_kps.append(np.zeros((17, 3), dtype=np.float32))
            all_boxes.append(np.zeros((4,), dtype=np.float32))
            all_valid.append(False)

            if f_idx < n_bb and bool(is_det[f_idx]):
                s0, s1 = int(non_pat_off[f_idx]), int(non_pat_off[f_idx + 1])
                sup_b = non_pat_flat[s0:s1, :4] if s1 > s0 else np.empty((0, 4), dtype=np.float32)
                crop, off_xy, core_box, pbox_px = _crop_and_mask_patient(
                    frame, pboxes_norm[f_idx], sup_b, cfg["crop_pad_ratio"], cfg["mask_supervisors"]
                )
                batch_crops.append(crop)
                batch_meta.append((f_idx, off_xy, core_box, pbox_px))
                if len(batch_crops) == cfg["batch_size"]:
                    flush_batch()
            f_idx += 1

        flush_batch()
        cap.release()

        np.savez_compressed(
            save_path,
            kps=np.asarray(all_kps, dtype=np.float32),
            boxes=np.asarray(all_boxes, dtype=np.float32),
            valid=np.asarray(all_valid, dtype=bool),
            fps=np.float32(fps),
            W=np.int32(W),
            H=np.int32(H),
        )
        print(f"[GPU {gpu_label}] Selesai {split}/{tid}: {len(all_kps)} frame | valid={np.mean(all_valid)*100:.2f}%", flush=True)


if __name__ == "__main__":
    main()
""", encoding="utf-8")

t0 = time.time()
n_gpus = max(1, torch.cuda.device_count())
tasks_list = df_manifest.to_dict("records")
cfg_dict = {
    "weights": str(Path(YOLO_WEIGHTS).resolve()) if Path(YOLO_WEIGHTS).exists() else YOLO_WEIGHTS,
    "out_dir": str(RAW_POSE_DIR),
    "batch_size": BATCH_SIZE,
    "imgsz": IMG_SIZE,
    "det_conf": DET_CONF,
    "crop_pad_ratio": CROP_PAD_RATIO,
    "mask_supervisors": MASK_SUPERVISORS,
    "min_crop_iou": MIN_CROP_IOU,
}

shards = [tasks_list[i::n_gpus] for i in range(n_gpus)]
procs = []
for gid in range(n_gpus):
    job_file = OUTPUT_ROOT / f"_yolo_job_gpu{gid}.json"
    job_file.write_text(json.dumps({"gpu_label": gid, "cfg": cfg_dict, "tasks": shards[gid]}), encoding="utf-8")
    env = os.environ.copy()
    if torch.cuda.is_available():
        env["CUDA_VISIBLE_DEVICES"] = str(gid)
    p = subprocess.Popen([sys.executable, str(worker_script), str(job_file)], env=env)
    procs.append((gid, p))

for gid, p in procs:
    rc = p.wait()
    assert rc == 0, f"Worker GPU {gid} gagal dengan exitcode {rc}"

print(f"Ekstraksi 71 video (PatientLock + YOLO26n-Pose) selesai dalam {time.time() - t0:.1f} detik.")


In [ ]:
# ==============================================================================
# TAHAP 2: FUNGSI PREPROCESSING SCENARIO A & SCENARIO B (LB6)
# ==============================================================================
from scipy.signal import butter, sosfiltfilt


def interpolate_low_conf(kps_lb6: np.ndarray, conf_thres: float = 0.20) -> np.ndarray:
    """Interpolasi linear koordinat (x, y) jika confidence < conf_thres."""
    out = kps_lb6.copy()
    T, V, _ = out.shape
    t_idx = np.arange(T)
    for v in range(V):
        good = out[:, v, 2] >= conf_thres
        if good.sum() == 0:
            out[:, v, :2] = 0.0
        elif good.sum() < T:
            for c in (0, 1):
                out[:, v, c] = np.interp(t_idx, t_idx[good], out[good, v, c])
    return out


def hampel_filter_1d(sig: np.ndarray, window: int = 7, n_sigma: float = 3.0) -> np.ndarray:
    """Hampel rolling-median filter untuk membersihkan lonjakan outlier pose."""
    s = pd.Series(sig, dtype=np.float64)
    med = s.rolling(window=window, center=True, min_periods=1).median()
    mad = (s - med).abs().rolling(window=window, center=True, min_periods=1).median()
    thresh = n_sigma * 1.4826 * mad
    out = s.copy()
    outlier = (s - med).abs() > thresh
    out[outlier] = med[outlier]
    return out.to_numpy(dtype=np.float32)


def butter_lowpass(sig: np.ndarray, fs: float, cutoff: float = 9.0, order: int = 4) -> np.ndarray:
    """Butterworth zero-phase low-pass filter (dipakai pada 30 Hz, cutoff < Nyquist 15 Hz)."""
    if cutoff >= 0.49 * fs or len(sig) < 18:
        return sig.astype(np.float32)
    sos = butter(order, cutoff, btype="low", fs=fs, output="sos")
    return sosfiltfilt(sos, sig).astype(np.float32)


def normalize_lb6(kps_lb6: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Pusatkan ke median pelvis & skalakan dengan median panjang kaki per trial."""
    # Indeks LB6: 0:L_Hip, 1:R_Hip, 2:L_Knee, 3:R_Knee, 4:L_Ankle, 5:R_Ankle
    pelvis = 0.5 * (kps_lb6[:, 0, :2] + kps_lb6[:, 1, :2])
    pelvis_center = np.median(pelvis, axis=0, keepdims=True)  # (1, 2)

    leg_l = (
        np.linalg.norm(kps_lb6[:, 0, :2] - kps_lb6[:, 2, :2], axis=-1)
        + np.linalg.norm(kps_lb6[:, 2, :2] - kps_lb6[:, 4, :2], axis=-1)
    )
    leg_r = (
        np.linalg.norm(kps_lb6[:, 1, :2] - kps_lb6[:, 3, :2], axis=-1)
        + np.linalg.norm(kps_lb6[:, 3, :2] - kps_lb6[:, 5, :2], axis=-1)
    )
    scale = float(np.median(0.5 * (leg_l + leg_r)))
    scale = max(scale, 1e-4)

    xy_norm = (kps_lb6[:, :, :2] - pelvis_center[:, None, :]) / scale
    conf = kps_lb6[:, :, 2:3]

    # Vektor tulang (bone_x, bone_y): Hip->pelvis(0), Knee->Hip, Ankle->Knee
    parent = [None, None, 0, 1, 2, 3]
    bones = np.zeros_like(xy_norm)
    for v, p in enumerate(parent):
        if p is not None:
            bones[:, v, :] = xy_norm[:, v, :] - xy_norm[:, p, :]
    return xy_norm.astype(np.float32), conf.astype(np.float32), bones.astype(np.float32)


def build_scenario_a_and_b(kps_17: np.ndarray, fs: float) -> tuple[np.ndarray, np.ndarray]:
    """Bangun Scenario A (T, 6, 7) dan Scenario B (T, 6, 9) untuk satu trial."""
    kps_lb6 = kps_17[:, LB6_INDICES, :].copy()
    kps_interp = interpolate_low_conf(kps_lb6, CONF_VALID_THRES)

    # --- Scenario A: [x, y, conf, dx, dy, bone_x, bone_y] ---
    xy_a, conf_a, bones_a = normalize_lb6(kps_interp)
    dx_a = np.gradient(xy_a[:, :, 0], axis=0)[:, :, None]
    dy_a = np.gradient(xy_a[:, :, 1], axis=0)[:, :, None]
    scen_a = np.concatenate([xy_a, conf_a, dx_a, dy_a, bones_a], axis=-1).astype(np.float32)

    # --- Scenario B: Hampel + Butterworth 9 Hz (pada 30 Hz) + accel ---
    kps_den = kps_interp.copy()
    for v in range(6):
        for c in (0, 1):
            sig = hampel_filter_1d(kps_den[:, v, c], window=HAMPEL_WINDOW, n_sigma=HAMPEL_SIGMA)
            if fs >= 25.0:
                sig = butter_lowpass(sig, fs=fs, cutoff=BUTTER_CUTOFF_HZ, order=4)
            kps_den[:, v, c] = sig

    xy_b, conf_b, bones_b = normalize_lb6(kps_den)
    dx_b = np.gradient(xy_b[:, :, 0], axis=0)[:, :, None]
    dy_b = np.gradient(xy_b[:, :, 1], axis=0)[:, :, None]
    ddx_b = np.gradient(dx_b[:, :, 0], axis=0)[:, :, None]
    ddy_b = np.gradient(dy_b[:, :, 0], axis=0)[:, :, None]
    scen_b = np.concatenate([xy_b, conf_b, dx_b, dy_b, ddx_b, ddy_b, bones_b], axis=-1).astype(np.float32)

    return scen_a, scen_b


In [ ]:
# ==============================================================================
# TAHAP 3: SIMPAN SCENARIO A, B, DAN D (WINDOWED 15 HZ & 30 HZ)
# ==============================================================================
SCEN_DIR = OUTPUT_ROOT / "02_processed_scenarios"


def process_frequency(target_hz: int):
    assert target_hz in (15, 30)
    step = 2 if target_hz == 15 else 1
    win_len = int(round(WINDOW_SEC * target_hz))     # 30 @ 15Hz | 60 @ 30Hz
    stride_len = int(round(STRIDE_SEC * target_hz))  # 15 @ 15Hz | 30 @ 30Hz

    for scen_name in ["A_baseline_norm", "B_denoised_hampel_accel"]:
        (SCEN_DIR / scen_name / f"yolo26n_lb6_{target_hz}hz").mkdir(parents=True, exist_ok=True)

    # Kumpulkan window Scenario D (dari fitur Scenario B sebagai default unggulan Rhama, dan A juga)
    for src_scen in ["A_baseline_norm", "B_denoised_hampel_accel"]:
        d_root = SCEN_DIR / "D_windowed_stgcn_cls" / f"from_{src_scen}_{target_hz}hz"
        d_root.mkdir(parents=True, exist_ok=True)

        split_buffers = {sp: {"X": [], "y_any": [], "y_maj": [], "y_ge25": [], "y_ratio": [], "trial_id": [], "subject_id": []} for sp in SPLITS}

        for row in df_manifest.itertuples():
            raw_npz = np.load(RAW_POSE_DIR / row.split / f"{row.trial_id}.npz")
            kps_all = raw_npz["kps"]  # (T30, 17, 3)

            lbl_path = row.label15_path if target_hz == 15 else row.label30_path
            df_lbl = pd.read_csv(lbl_path)

            kps_hz = kps_all[::step]
            n_frames = min(len(kps_hz), len(df_lbl))
            kps_hz = kps_hz[:n_frames]
            df_lbl = df_lbl.iloc[:n_frames].reset_index(drop=True)

            if src_scen == "A_baseline_norm":
                feat, _ = build_scenario_a_and_b(kps_hz, fs=float(target_hz))
                scen_save = SCEN_DIR / "A_baseline_norm" / f"yolo26n_lb6_{target_hz}hz" / row.split
            else:
                _, feat = build_scenario_a_and_b(kps_hz, fs=float(target_hz))
                scen_save = SCEN_DIR / "B_denoised_hampel_accel" / f"yolo26n_lb6_{target_hz}hz" / row.split

            scen_save.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(
                scen_save / f"{row.trial_id}.npz",
                X=feat,
                y_fog=df_lbl["y_fog"].to_numpy(dtype=np.int64),
                inside_imu_mask=df_lbl["inside_imu_mask"].to_numpy(dtype=np.int64),
            )

            # Potong sliding window HANYA di frame dengan inside_imu_mask == 1
            mask = df_lbl["inside_imu_mask"].to_numpy(dtype=np.int64)
            y_seq = df_lbl["y_fog"].to_numpy(dtype=np.int64)

            for s_idx in range(0, n_frames - win_len + 1, stride_len):
                e_idx = s_idx + win_len
                if np.any(mask[s_idx:e_idx] == 0):
                    continue
                w_feat = feat[s_idx:e_idx]                # (T_win, 6, C)
                w_tensor = np.transpose(w_feat, (2, 0, 1)) # (C, T_win, 6) untuk ST-GCN / Mamba
                fog_ratio = float(np.mean(y_seq[s_idx:e_idx]))

                buf = split_buffers[row.split]
                buf["X"].append(w_tensor)
                buf["y_any"].append(int(fog_ratio > 0.0))
                buf["y_maj"].append(int(fog_ratio >= 0.5))
                buf["y_ge25"].append(int(fog_ratio >= 0.25))
                buf["y_ratio"].append(fog_ratio)
                buf["trial_id"].append(row.trial_id)
                buf["subject_id"].append(row.subject_id)

        # Fit StandardScaler HANYA pada train (Zero Data Leakage!)
        X_train = np.asarray(split_buffers["train"]["X"], dtype=np.float32)  # (N, C, T, 6)
        mean = X_train.mean(axis=(0, 2, 3), keepdims=True)
        std = X_train.std(axis=(0, 2, 3), keepdims=True) + 1e-6
        np.savez(d_root / "scaler_stats.npz", mean=mean, std=std)

        # Buat indeks train seimbang 1:1 (y_maj)
        rng = np.random.default_rng(RANDOM_SEED)
        y_tr_maj = np.asarray(split_buffers["train"]["y_maj"], dtype=np.int64)
        pos_idx = np.where(y_tr_maj == 1)[0]
        neg_idx = np.where(y_tr_maj == 0)[0]
        n_bal = min(len(pos_idx), len(neg_idx))
        bal_idx = np.concatenate([
            rng.choice(pos_idx, size=n_bal, replace=False),
            rng.choice(neg_idx, size=n_bal, replace=False),
        ])
        rng.shuffle(bal_idx)

        for sp in SPLITS:
            buf = split_buffers[sp]
            X_sp = np.asarray(buf["X"], dtype=np.float32)
            assert not np.isnan(X_sp).any() and not np.isinf(X_sp).any(), f"NaN/Inf ditemukan di {sp}"
            np.savez_compressed(
                d_root / f"{sp}.npz",
                X=X_sp,
                X_scaled=((X_sp - mean) / std).astype(np.float32),
                y_any=np.asarray(buf["y_any"], dtype=np.int64),
                y_maj=np.asarray(buf["y_maj"], dtype=np.int64),
                y_ge25=np.asarray(buf["y_ge25"], dtype=np.int64),
                y_ratio=np.asarray(buf["y_ratio"], dtype=np.float32),
                trial_id=np.asarray(buf["trial_id"]),
                subject_id=np.asarray(buf["subject_id"]),
                balanced_train_idx=bal_idx if sp == "train" else np.array([], dtype=np.int64),
            )
            print(f"[{src_scen} @ {target_hz}Hz] {sp:5s}: X={X_sp.shape} | FoG(maj)={np.mean(buf['y_maj'])*100:.2f}%")


process_frequency(15)
process_frequency(30)


In [ ]:
# ==============================================================================
# TAHAP 4: SMOKE TEST PYTORCH DATASET & MATRIKS GRAF LB6 (SIAP TRAINING!)
# ==============================================================================
from torch.utils.data import Dataset, DataLoader


def build_lb6_graph_A() -> np.ndarray:
    """Matriks ketetanggaan spasial A (3, 6, 6): Root, Centripetal, Centrifugal."""
    V = 6
    # 0:L_Hip, 1:R_Hip, 2:L_Knee, 3:R_Knee, 4:L_Ankle, 5:R_Ankle
    self_link = np.eye(V, dtype=np.float32)
    inward_edges = [(2, 0), (3, 1), (4, 2), (5, 3), (0, 1)]  # menuju pusat pelvis
    inward = np.zeros((V, V), dtype=np.float32)
    outward = np.zeros((V, V), dtype=np.float32)
    for i, j in inward_edges:
        inward[j, i] = 1.0
        outward[i, j] = 1.0

    def norm_adj(M):
        deg = M.sum(axis=0, keepdims=True)
        deg[deg == 0] = 1.0
        return M / deg

    return np.stack([norm_adj(self_link), norm_adj(inward), norm_adj(outward)], axis=0)


class MobileFogWindowDataset(Dataset):
    def __init__(self, npz_path: Path, label_key: str = "y_maj", use_scaled: bool = True, indices: np.ndarray | None = None):
        data = np.load(npz_path)
        self.X = data["X_scaled"] if use_scaled else data["X"]
        self.y = data[label_key]
        if indices is not None and len(indices) > 0:
            self.X = self.X[indices]
            self.y = self.y[indices]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)


test_npz = SCEN_DIR / "D_windowed_stgcn_cls" / "from_B_denoised_hampel_accel_30hz" / "train.npz"
ds_tr = MobileFogWindowDataset(test_npz, label_key="y_maj", use_scaled=True)
dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
xb, yb = next(iter(dl_tr))
A_lb6 = build_lb6_graph_A()

print(f"Batch X shape : {tuple(xb.shape)}  (B, C, T, V)")
print(f"Batch y shape : {tuple(yb.shape)}")
print(f"Graf LB6 A    : {A_lb6.shape}")
print("Semua output siap di /kaggle/working/mobilefog_rhama/ untuk training ST-GCN-cls, Bi-Mamba, atau Hybrid!")
